In [8]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


# 1. DATA WRANGLING

df = pd.read_csv("raw_data.csv")

# Removing the Country Code from the raw_data.
# Example: au Australia to Australia. This make the t-test and the results of raw data even more smoothier
#        and gives the data as required to the assessment.

df["Squad"] = df["Squad"].str.split(" ", n=1).str[1]

# Confederation mapping (confirmed for the 2026 World Cup)
uefa_teams = ["Spain", "France", "England", "Germany", "Netherlands", "Belgium",
              "Portugal", "Croatia", "Switzerland", "Austria", "Norway", "Scotland",
              "Czechia", "Türkiye", "Sweden", "Bosnia–Herz"]

conmebol_teams = ["Argentina", "Brazil", "Uruguay", "Colombia", "Ecuador", "Paraguay"]

def tag_confederation(team):
    if team in uefa_teams:
        return "UEFA"
    elif team in conmebol_teams:
        return "CONMEBOL"
    return None

df["Confederation"] = df["Squad"].apply(tag_confederation)

# Filtering the teams into two groups. As we need just 2 groups.
analysis_df = df[df["Confederation"].isin(["UEFA", "CONMEBOL"])].copy()

# Analysing the Conversion Rate of Shots (goals per shot) -- matches from FBref's G/Sh
analysis_df["ConversionRate"] = analysis_df["Gls"] / analysis_df["Sh"]

print("=" * 70)
print("WRANGLED DATA (UEFA + CONMEBOL only)")
print("=" * 70)
print(analysis_df[["Squad", "Confederation", "Gls", "Sh", "ConversionRate"]]
      .sort_values(["Confederation", "ConversionRate"], ascending=[True, False])
      .to_string(index=False))

uefa = analysis_df.loc[analysis_df["Confederation"] == "UEFA", "ConversionRate"]
conmebol = analysis_df.loc[analysis_df["Confederation"] == "CONMEBOL", "ConversionRate"]

# ---------------------------------------------------------------
# 2. DATA PREPARATION & SAMPLING
# ---------------------------------------------------------------
# Population: the long-term/underlying attacking effectiveness of national 
#            teams associated with CONMEBOL and UEFA, correspondingly (not 
#            just the result of this particular championship, which is loud in and of itself).
# Sample: The World Cup 2026 conversion rate of each of the 16 UEFA and 6 
#         CONMEBOL teams which reached the World Cup stage. It is a 
#         CENSUS of the WC26 qualifiers of the confederation, and is 
#         being used here as a sample of the confederation's team attack 
#         capability because all the qualifying teams have been considered.
# Sampling technique: It is not a random sample but an exhaustive study 
#         (census) of the entire sampling frame (confederations’ teams 
#         that qualified for the World Cup of 2026). This is highlighted 
#         as a limitation in the paper, where inference is described as 
#         “how the entrants of these confederations performed.”
print("\n" + "=" * 70)
print("SAMPLE SIZES")
print("=" * 70)
print(f"UEFA:     n = {len(uefa)}")
print(f"CONMEBOL: n = {len(conmebol)}")


# 3. DESCRIPTIVE STATISTICS

def describe(sample, label):
    return {
        "Group": label,
        "n": len(sample),
        "Mean": sample.mean(),
        "Median": sample.median(),
        "Std Dev": sample.std(ddof=1),
        "Min": sample.min(),
        "Max": sample.max(),
    }

desc = pd.DataFrame([describe(uefa, "UEFA"), describe(conmebol, "CONMEBOL")])
print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)
print(desc.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# 4. INFERENTIAL STATISTICS -- CONFIDENCE INTERVALS

def ci_mean(sample, confidence=0.95):
    n = len(sample)
    mean = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    margin = t_crit * se
    return mean, mean - margin, mean + margin

uefa_mean, uefa_lo, uefa_hi = ci_mean(uefa)
conmebol_mean, conmebol_lo, conmebol_hi = ci_mean(conmebol)

print("\n" + "=" * 70)
print("95% CONFIDENCE INTERVALS FOR MEAN CONVERSION RATE")
print("=" * 70)
print(f"UEFA:     mean = {uefa_mean:.4f}, 95% CI = [{uefa_lo:.4f}, {uefa_hi:.4f}]")
print(f"CONMEBOL: mean = {conmebol_mean:.4f}, 95% CI = [{conmebol_lo:.4f}, {conmebol_hi:.4f}]")

# CI for the DIFFERENCE in means (Welch, unequal variances)
n1, n2 = len(uefa), len(conmebol)
m1, m2 = uefa.mean(), conmebol.mean()
s1, s2 = uefa.std(ddof=1), conmebol.std(ddof=1)
se_diff = np.sqrt(s1**2 / n1 + s2**2 / n2)
diff = m1 - m2

# Welch-Satterthwaite degrees of freedom
dof = (s1**2 / n1 + s2**2 / n2)**2 / (
    (s1**2 / n1)**2 / (n1 - 1) + (s2**2 / n2)**2 / (n2 - 1)
)
t_crit_diff = stats.t.ppf(0.975, df=dof)
diff_lo = diff - t_crit_diff * se_diff
diff_hi = diff + t_crit_diff * se_diff

print(f"\nDifference in means (UEFA - CONMEBOL) = {diff:.4f}")
print(f"95% CI for the difference = [{diff_lo:.4f}, {diff_hi:.4f}]")
print(f"(Welch-Satterthwaite df = {dof:.2f})")


# 5. INFERENTIAL STATISTICS -- TWO-SAMPLE T-TEST

# Welch's t-test (does NOT assume equal variances) -- the safer default,
# especially with unequal and small sample sizes (n=16 vs n=6).
t_stat, p_value = stats.ttest_ind(uefa, conmebol, equal_var=False)

# t-test comparison for the selected question for my part.
t_stat_eq, p_value_eq = stats.ttest_ind(uefa, conmebol, equal_var=True)

print("\n" + "=" * 70)
print("TWO-SAMPLE T-TEST: UEFA vs CONMEBOL conversion rate")
print("=" * 70)
print("H0: mu_UEFA = mu_CONMEBOL")
print("H1: mu_UEFA != mu_CONMEBOL")
print(f"\nWelch's t-test (unequal variances assumed):")
print(f"  t = {t_stat:.4f}, df = {dof:.2f}, p = {p_value:.4f}")
print(f"\nStudent's t-test (equal variances assumed, for comparison):")
print(f"  t = {t_stat_eq:.4f}, df = {n1+n2-2}, p = {p_value_eq:.4f}")

alpha = 0.05
print(f"\nAt alpha = {alpha}:")
if p_value < alpha:
    print("  Reject H0 -> statistically significant difference in conversion rate.")
else:
    print("  Fail to reject H0 -> no statistically significant difference detected.")


# 6. ROBUSTNESS CHECK -- Mann-Whitney U (non-parametric)

# Given the small CONMEBOL sample (n=6), a non-parametric check is a useful
# sanity check since it doesn't rely on a normality assumption.
u_stat, p_mw = stats.mannwhitneyu(uefa, conmebol, alternative="two-sided")
print("\n" + "=" * 70)
print("ROBUSTNESS CHECK: Mann-Whitney U test")
print("=" * 70)
print(f"U = {u_stat:.4f}, p = {p_mw:.4f}")

# Shapiro-Wilk normality checks (informational)
print("\n" + "=" * 70)
print("NORMALITY CHECK (Shapiro-Wilk, informational)")
print("=" * 70)
sw_uefa = stats.shapiro(uefa)
sw_conmebol = stats.shapiro(conmebol)
print(f"UEFA:     W = {sw_uefa.statistic:.4f}, p = {sw_uefa.pvalue:.4f}")
print(f"CONMEBOL: W = {sw_conmebol.statistic:.4f}, p = {sw_conmebol.pvalue:.4f}")

# Save cleaned data for reuse
analysis_df.to_csv("cleaned_analysis_data.csv", index=False)
print("\nCleaned dataset saved to cleaned_analysis_data.csv")

WRANGLED DATA (UEFA + CONMEBOL only)
      Squad Confederation  Gls  Sh  ConversionRate
  Argentina      CONMEBOL   18 114        0.157895
     Brazil      CONMEBOL   10  74        0.135135
   Paraguay      CONMEBOL    3  35        0.085714
    Uruguay      CONMEBOL    3  49        0.061224
   Colombia      CONMEBOL    5  94        0.053191
    Ecuador      CONMEBOL    2  53        0.037736
Netherlands          UEFA   10  46        0.217391
     Norway          UEFA   12  66        0.181818
    England          UEFA   20 118        0.169492
    Croatia          UEFA    6  37        0.162162
    Austria          UEFA    5  32        0.156250
    Germany          UEFA   11  74        0.148649
     Sweden          UEFA    7  48        0.145833
     France          UEFA   20 139        0.143885
Switzerland          UEFA   10  74        0.135135
    Belgium          UEFA   13 112        0.116071
   Portugal          UEFA    7  62        0.112903
Bosnia–Herz          UEFA    4  37        0.1